# 01 - Validate Agent Blueprint Configuration

**Learning objectives**
- Validate `.env` configuration generated by `a365.ps1` script
- Understand the components of an Agent Identity Blueprint setup
- Verify connectivity to Azure Entra ID using the configured credentials
- Test both client secret and certificate-based authentication methods
- Prepare for token acquisition and agent identity operations

**Prerequisites**
- Python 3.11+ (3.13 recommended)
- Completed `a365.ps1` setup (run with `-Step menu` or `-Step all`)
- `.env` file populated with Azure credentials
- Microsoft Graph PowerShell SDK installed (for PowerShell script)

**How to use this notebook**
- Run cells from top to bottom
- First validates your configuration from `.env`
- Then tests authentication with your chosen credential type
- Provides diagnostics and troubleshooting guidance

## What is an Agent Identity Blueprint?

An **Agent Identity Blueprint** in Microsoft Entra ID is a template that defines:
- **Identity**: A service principal (application) registered in your Azure tenant
- **Authentication**: Client credentials (secret or certificate) for programmatic access
- **Permissions**: OAuth scopes that control what APIs the agent can access
- **Scope URI**: The identifier URI that other apps use to request tokens for this agent

The `a365.ps1` script creates this infrastructure in 4 steps:
1. **Create Blueprint**: Registers the application in Entra ID
2. **Create Service Principal**: Instantiates the identity in the directory
3. **Add Credentials**: Configures authentication (secret or certificate)
4. **Configure OAuth Scope**: Sets up the API permissions and identifier URI

**Key concepts**
- **Application (Client) ID**: Unique identifier for your agent blueprint (used in token requests)
- **Tenant ID**: Your Azure AD directory ID (determines which tenant the agent belongs to)
- **Client Secret**: A password-like credential for authentication (easier but less secure)
- **Client Certificate**: A certificate-based credential (more secure, recommended for production)
- **Identifier URI**: API identifier in format `api://{clientId}` used for permission delegation
- **OAuth Scope**: Permission scope (`access_agent`) that other apps must request

**Documentation references**
- [Agent Identity Blueprint overview](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/agent-blueprint)
- [Create agent blueprint](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/create-blueprint)
- [Service principals and applications](https://learn.microsoft.com/en-us/entra/identity-platform/app-objects-and-service-principals)

## Step 1: Load and Validate Environment Configuration

The `a365.ps1` script stores all configuration in a `.env` file. Let's load it and verify all required values are present.

**Expected variables:**
- `AZURE_TENANT_ID`: Your Azure AD tenant ID
- `AZURE_CLIENT_ID`: Application (client) ID of the blueprint
- `AZURE_CLIENT_CREDENTIAL_TYPE`: Either `secret` or `certificate`
- `AZURE_CLIENT_SECRET`: Client secret value (if using secret authentication)
- `AZURE_CLIENT_SECRET_KEY_ID`: Key ID of the secret (for reference)
- `AZURE_CLIENT_CERT_PATH`: Path to certificate file (if using certificate)
- `AZURE_CLIENT_CERT_THUMBPRINT`: Certificate thumbprint
- `AGENT_BLUEPRINT_ID`: Object ID of the blueprint application
- `AGENT_BLUEPRINT_PRINCIPAL_ID`: Service principal object ID
- `AGENT_BLUEPRINT_IDENTIFIER_URI`: OAuth identifier URI
- `AGENT_BLUEPRINT_SCOPE`: OAuth scope name

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
env_path = Path(".env")
if not env_path.exists():
    raise FileNotFoundError(
        "❌ .env file not found! Run the a365.ps1 script first to generate configuration.\n"
        "   Usage: pwsh a365.ps1 -Step menu"
    )

load_dotenv(env_path)
print("✅ Loaded .env file\n")

# Define required variables and their purposes
required_vars = {
    "AZURE_TENANT_ID": "Azure AD tenant identifier",
    "AZURE_CLIENT_ID": "Application (client) ID of the blueprint",
    "AZURE_CLIENT_CREDENTIAL_TYPE": "Credential type (secret or certificate)",
    "AGENT_BLUEPRINT_ID": "Blueprint application object ID",
    "AGENT_BLUEPRINT_PRINCIPAL_ID": "Service principal object ID",
    "AGENT_BLUEPRINT_IDENTIFIER_URI": "OAuth identifier URI",
    "AGENT_BLUEPRINT_SCOPE": "OAuth permission scope"
}

# Validate required variables
config = {}
missing = []
for var, description in required_vars.items():
    value = os.getenv(var)
    if not value:
        missing.append(f"  - {var}: {description}")
    else:
        config[var] = value

if missing:
    raise ValueError(
        f"❌ Missing required environment variables:\n" +
        "\n".join(missing) +
        "\n\nRun the a365.ps1 script to complete all 4 steps."
    )

# Get credential type and validate corresponding credentials
credential_type = config["AZURE_CLIENT_CREDENTIAL_TYPE"]
print(f"📋 Configuration Summary:")
print(f"   Tenant ID: {config['AZURE_TENANT_ID']}")
print(f"   Client ID: {config['AZURE_CLIENT_ID']}")
print(f"   Credential Type: {credential_type}")
print(f"   Blueprint ID: {config['AGENT_BLUEPRINT_ID']}")
print(f"   Principal ID: {config['AGENT_BLUEPRINT_PRINCIPAL_ID']}")
print(f"   Identifier URI: {config['AGENT_BLUEPRINT_IDENTIFIER_URI']}")
print(f"   OAuth Scope: {config['AGENT_BLUEPRINT_SCOPE']}\n")

# Validate credential-specific configuration
if credential_type == "secret":
    client_secret = os.getenv("AZURE_CLIENT_SECRET")
    secret_key_id = os.getenv("AZURE_CLIENT_SECRET_KEY_ID")
    if not client_secret:
        raise ValueError("❌ AZURE_CLIENT_SECRET is required for secret-based authentication")
    print(f"🔑 Client Secret Configuration:")
    print(f"   Secret Key ID: {secret_key_id}")
    print(f"   Secret Value: {'*' * 20} (hidden)")
    config["AZURE_CLIENT_SECRET"] = client_secret
    
elif credential_type == "certificate":
    cert_path = os.getenv("AZURE_CLIENT_CERT_PATH")
    cert_thumbprint = os.getenv("AZURE_CLIENT_CERT_THUMBPRINT")
    if not cert_path or not cert_thumbprint:
        raise ValueError(
            "❌ AZURE_CLIENT_CERT_PATH and AZURE_CLIENT_CERT_THUMBPRINT are required "
            "for certificate-based authentication"
        )
    cert_file = Path(cert_path)
    if not cert_file.exists():
        raise FileNotFoundError(f"❌ Certificate file not found: {cert_path}")
    print(f"📜 Certificate Configuration:")
    print(f"   Certificate Path: {cert_path}")
    print(f"   Certificate Thumbprint: {cert_thumbprint}")
    print(f"   File Size: {cert_file.stat().st_size} bytes")
    config["AZURE_CLIENT_CERT_PATH"] = cert_path
    config["AZURE_CLIENT_CERT_THUMBPRINT"] = cert_thumbprint
else:
    raise ValueError(
        f"❌ Invalid credential type: {credential_type}. "
        "Expected 'secret' or 'certificate'"
    )

print("\n✅ Configuration validation complete!")

## Step 2: Test Authentication

Now let's verify we can authenticate to Azure Entra ID using the configured credentials.

**What this does:**
1. Uses the Microsoft Authentication Library (MSAL) to acquire an access token
2. Authenticates using either client secret or certificate (based on your configuration)
3. Requests a token for Microsoft Graph API (`https://graph.microsoft.com/.default`)
4. Validates the token and extracts claims

**Authentication flows:**
- **Client Secret**: Simple password-based authentication (OAuth 2.0 client credentials flow)
- **Client Certificate**: More secure certificate-based authentication (client assertion)

**Reference:** [OAuth 2.0 client credentials flow](https://learn.microsoft.com/en-us/entra/identity-platform/v2-oauth2-client-creds-grant-flow)

In [ ]:
import msal
import jwt
from datetime import datetime, timezone

# Build MSAL authority URL
authority = f"https://login.microsoftonline.com/{config['AZURE_TENANT_ID']}"
print(f"🔐 Authenticating to: {authority}\n")

# Configure client credentials based on type
if credential_type == "secret":
    print("🔑 Using client secret authentication...")
    app = msal.ConfidentialClientApplication(
        client_id=config["AZURE_CLIENT_ID"],
        client_credential=config["AZURE_CLIENT_SECRET"],
        authority=authority
    )
elif credential_type == "certificate":
    print("📜 Using client certificate authentication...")
    # Read private key from certificate file
    with open(config["AZURE_CLIENT_CERT_PATH"], "r") as f:
        private_key = f.read()
    
    client_credential = {
        "private_key": private_key,
        "thumbprint": config["AZURE_CLIENT_CERT_THUMBPRINT"]
    }
    app = msal.ConfidentialClientApplication(
        client_id=config["AZURE_CLIENT_ID"],
        client_credential=client_credential,
        authority=authority
    )

# Acquire token for Microsoft Graph
scopes = ["https://graph.microsoft.com/.default"]
print(f"🎫 Requesting token for scopes: {', '.join(scopes)}\n")

try:
    result = app.acquire_token_for_client(scopes=scopes)
    
    if "access_token" in result:
        access_token = result["access_token"]
        expires_in = result.get("expires_in", 0)
        
        print("✅ Authentication successful!\n")
        print(f"📊 Token Information:")
        print(f"   Token Type: Bearer")
        print(f"   Expires In: {expires_in} seconds ({expires_in // 60} minutes)")
        print(f"   Token Length: {len(access_token)} characters")
        print(f"   Token Preview: {access_token[:50]}...\n")
        
        # Decode token to extract claims (without validation for inspection)
        # Note: In production, always validate tokens properly
        decoded = jwt.decode(access_token, options={"verify_signature": False})
        
        print(f"🏷️  Token Claims:")
        print(f"   Issuer: {decoded.get('iss', 'N/A')}")
        print(f"   Audience: {decoded.get('aud', 'N/A')}")
        print(f"   Subject: {decoded.get('sub', 'N/A')}")
        print(f"   Application ID: {decoded.get('appid', 'N/A')}")
        print(f"   Tenant ID: {decoded.get('tid', 'N/A')}")
        
        # Check if our client ID matches the token
        if decoded.get('appid') == config['AZURE_CLIENT_ID']:
            print(f"   ✅ Token appid matches configured AZURE_CLIENT_ID")
        else:
            print(f"   ⚠️  Token appid does not match configured AZURE_CLIENT_ID")
        
        # Check token expiration
        exp_timestamp = decoded.get('exp')
        if exp_timestamp:
            exp_datetime = datetime.fromtimestamp(exp_timestamp, tz=timezone.utc)
            print(f"   Expires At: {exp_datetime.isoformat()}")
        
        # Store token for next notebook
        config['ACCESS_TOKEN'] = access_token
        
    else:
        error = result.get("error")
        error_description = result.get("error_description")
        print(f"❌ Authentication failed!\n")
        print(f"   Error: {error}")
        print(f"   Description: {error_description}")
        raise Exception(f"Authentication failed: {error}")
        
except Exception as e:
    print(f"❌ Exception during authentication: {e}")
    raise

## Step 3: Verify Microsoft Graph Connectivity

Let's make a simple Microsoft Graph API call to verify our token works and the blueprint has proper permissions.

We'll query the service principal to confirm it exists and retrieve its properties.

**Endpoint:** `GET /v1.0/servicePrincipals/{id}`

In [ ]:
import requests

# Query our service principal from Microsoft Graph
principal_id = config['AGENT_BLUEPRINT_PRINCIPAL_ID']
url = f"https://graph.microsoft.com/v1.0/servicePrincipals/{principal_id}"

headers = {
    "Authorization": f"Bearer {config['ACCESS_TOKEN']}",
    "Content-Type": "application/json"
}

print(f"🌐 Testing Microsoft Graph connectivity...")
print(f"   URL: {url}\n")

try:
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        principal = response.json()
        print("✅ Microsoft Graph API call successful!\n")
        print(f"📋 Service Principal Information:")
        print(f"   Display Name: {principal.get('displayName')}")
        print(f"   Object ID: {principal.get('id')}")
        print(f"   Application ID: {principal.get('appId')}")
        print(f"   Service Principal Type: {principal.get('servicePrincipalType')}")
        print(f"   Account Enabled: {principal.get('accountEnabled')}")
        print(f"   App Display Name: {principal.get('appDisplayName')}")
        
        # Verify configuration matches
        if principal.get('id') == config['AGENT_BLUEPRINT_PRINCIPAL_ID']:
            print(f"\n   ✅ Service principal ID matches configuration")
        if principal.get('appId') == config['AZURE_CLIENT_ID']:
            print(f"   ✅ Application ID matches configuration")
        
        # Check identifier URIs
        identifier_uris = principal.get('identifierUris', [])
        if identifier_uris:
            print(f"\n   🔗 Identifier URIs:")
            for uri in identifier_uris:
                print(f"      - {uri}")
                if uri == config['AGENT_BLUEPRINT_IDENTIFIER_URI']:
                    print(f"        ✅ Matches configured AGENT_BLUEPRINT_IDENTIFIER_URI")
    
    elif response.status_code == 403:
        print("❌ Permission denied (403 Forbidden)\n")
        print("   Your application may not have the required permissions.")
        print("   Required permissions: Directory.Read.All or Application.Read.All")
        print("   Make sure you've granted admin consent in the Azure Portal.")
        
    elif response.status_code == 404:
        print("❌ Service principal not found (404 Not Found)\n")
        print(f"   The service principal ID may be incorrect: {principal_id}")
        print("   Re-run the a365.ps1 script to recreate the service principal.")
        
    else:
        print(f"❌ API call failed with status code: {response.status_code}\n")
        try:
            error = response.json()
            print(f"   Error: {error}")
        except:
            print(f"   Response: {response.text}")
            
except Exception as e:
    print(f"❌ Exception during API call: {e}")
    raise

## Configuration Summary

Let's create a summary JSON that captures all validated configuration for reference:

In [ ]:
# Build configuration summary (exclude sensitive data)
summary = {
    "tenant": {
        "id": config['AZURE_TENANT_ID'],
        "authority": authority
    },
    "blueprint": {
        "application_id": config['AZURE_CLIENT_ID'],
        "object_id": config['AGENT_BLUEPRINT_ID'],
        "service_principal_id": config['AGENT_BLUEPRINT_PRINCIPAL_ID'],
        "identifier_uri": config['AGENT_BLUEPRINT_IDENTIFIER_URI'],
        "oauth_scope": config['AGENT_BLUEPRINT_SCOPE']
    },
    "authentication": {
        "credential_type": credential_type,
        "certificate_path": config.get('AZURE_CLIENT_CERT_PATH', None),
        "certificate_thumbprint": config.get('AZURE_CLIENT_CERT_THUMBPRINT', None),
        "secret_key_id": os.getenv('AZURE_CLIENT_SECRET_KEY_ID', None)
    },
    "validation": {
        "config_loaded": True,
        "authentication_successful": 'ACCESS_TOKEN' in config,
        "graph_api_accessible": response.status_code == 200 if 'response' in locals() else False
    }
}

print("📊 Configuration Summary:\n")
print(json.dumps(summary, indent=2))

# Save to file for reference
with open("config-summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\n💾 Configuration summary saved to: config-summary.json")

## Troubleshooting Guide

### Common Issues and Solutions

**1. Missing .env file**
- **Problem**: `.env` file not found
- **Solution**: Run the `a365.ps1` script with `-Step menu` to generate configuration
  ```powershell
  pwsh a365.ps1 -Step menu
  ```

**2. Authentication fails with "invalid_client"**
- **Problem**: Client secret or certificate is incorrect
- **Solution**: Re-run Step 3 of `a365.ps1` to regenerate credentials
  ```powershell
  pwsh a365.ps1 -Step 3
  ```

**3. Certificate file not found**
- **Problem**: Certificate path in `.env` is incorrect
- **Solution**: Check `AZURE_CLIENT_CERT_PATH` points to the correct file in `./certs/` directory

**4. Graph API returns 403 Forbidden**
- **Problem**: Missing permissions or admin consent not granted
- **Solution**: 
  1. Go to Azure Portal → Entra ID → App registrations → your app
  2. Navigate to "API permissions"
  3. Verify required permissions are added
  4. Click "Grant admin consent" if not already granted

**5. Token claims show wrong application ID**
- **Problem**: Token was issued for a different application
- **Solution**: Verify `AZURE_CLIENT_ID` in `.env` matches the application ID you're using

**6. Service principal not found (404)**
- **Problem**: Service principal was deleted or ID is incorrect
- **Solution**: Re-run Step 2 of `a365.ps1` to recreate the service principal
  ```powershell
  pwsh a365.ps1 -Step 2
  ```

## Next Steps

✅ **Congratulations!** You've successfully validated your Agent Identity Blueprint configuration.

**What you've accomplished:**
- ✅ Loaded and validated `.env` configuration
- ✅ Successfully authenticated to Azure Entra ID
- ✅ Acquired an access token using your credentials
- ✅ Verified Microsoft Graph API connectivity
- ✅ Confirmed service principal exists and matches configuration

**Continue to the next notebook:**
- **[02-token-flows.ipynb](./02-token-flows.ipynb)**: Deep dive into token acquisition, examining different authentication flows and token types
- Learn about autonomous agent tokens vs. user-delegated tokens
- Understand token claims and how to use them for API access
- Test different OAuth 2.0 flows (client credentials, on-behalf-of)

**Additional resources:**
- [Microsoft Entra SDK for Agent Identities](https://learn.microsoft.com/en-us/entra/agent-id/identity-platform/microsoft-entra-sdk-for-agent-identities)
- [Agent Identity documentation](https://learn.microsoft.com/en-us/entra/agent-id/)
- [OAuth 2.0 in Microsoft identity platform](https://learn.microsoft.com/en-us/entra/identity-platform/v2-oauth2-auth-code-flow)